In [ ]:
from datasets import load_dataset, DatasetDict
from local_rag.core.text_cleaning import clean_text
from local_rag.utils.logger import get_logger
import time
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pathlib import Path


cwd = Path.cwd()
load_dotenv(cwd / "../../.env")
HF_TOKEN = os.getenv("HF_TOKEN")
login(HF_TOKEN)

logger = get_logger(__name__)
DATASET_NAME = "PrimeQA/clapnq_passages"
USER_NAME = "joshuale"
CLEANED_DATASET_NAME = f"{USER_NAME}/clapnq_passages_cleaned"
SAMPLE_SIZE = 1000

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
def clean_batch_corpus(batch):
    texts = batch["text"]
    cleaned_texts = []
    for text in texts:
        cleaned_text = clean_text(text)
        cleaned_texts.append(cleaned_text)
    return {"text": cleaned_texts}

## 1. Loading Corpus

In [3]:
# dataset = load_dataset(DATASET_NAME, split="train")
dataset_dict = load_dataset(DATASET_NAME)

logger.info(f"Available splits: {list(dataset_dict.keys())}")
logger.info(f"Train size: {len(dataset_dict['train'])}")

2026-01-09 00:29:34 INFO     Available splits: ['train']

                    INFO     Train size: 178890

## 2. Clean a Sample

In [4]:
sample_dataset = dataset_dict['train'].select(indices=range(SAMPLE_SIZE))

In [5]:
start = time.time()
cleaned_dataset = sample_dataset.map(
    clean_batch_corpus,
    batched=True,
    batch_size=200,
    num_proc=4,
)
end = time.time()
logger.info(f"Cleaning {SAMPLE_SIZE} samples took {end - start:.2f} seconds")

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

2026-01-09 00:30:33 INFO     Cleaning 1000 samples took 0.23 seconds

In [6]:
print("Original sample:")
print(sample_dataset[0].get('text'))

print("Cleaned sample:")
print(cleaned_dataset[0].get('text'))

Original sample:
Egocentrism is the inability to differentiate between self and other . More specifically , it is the inability to untangle subjective schemas from objective reality ; an inability to understand or assume any perspective other than their own .
Cleaned sample:
Egocentrism is the inability to differentiate between self and other. More specifically, it is the inability to untangle subjective schemas from objective reality; an inability to understand or assume any perspective other than their own.


## 3. Clean the Whole Dataset

In [8]:
start = time.time()
cleaned_dataset = dataset_dict['train'].map(
    clean_batch_corpus,
    batched=True,
    batch_size=1000,
    num_proc=4,
)
end = time.time()
logger.info(f"Cleaning {len(dataset_dict['train'])} train samples took {end - start:.2f} seconds")

Map (num_proc=4):   0%|          | 0/178890 [00:00<?, ? examples/s]

2026-01-09 00:33:01 INFO     Cleaning 178890 train samples took 6.44 seconds

## 4. Push to HF

In [9]:
cleaned_dataset_dict = DatasetDict({
    'train': cleaned_dataset,
})

logger.info(f"Pushing cleaned dataset to {CLEANED_DATASET_NAME}...")
cleaned_dataset_dict.push_to_hub(
    CLEANED_DATASET_NAME,
    private=True,          # if you don’t want it public
    max_shard_size="5GB",  # shard size for large data
)

2026-01-09 00:33:52 INFO     Pushing cleaned dataset to joshuale/clapnq_passages_cleaned...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/346 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/joshuale/clapnq_passages_cleaned/commit/973121c7de4a5a660d9ad196f6344124a039efac', commit_message='Upload dataset', commit_description='', oid='973121c7de4a5a660d9ad196f6344124a039efac', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/joshuale/clapnq_passages_cleaned', endpoint='https://huggingface.co', repo_type='dataset', repo_id='joshuale/clapnq_passages_cleaned'), pr_revision=None, pr_num=None)